In [0]:
%sql
-- ======================================================
-- DOMAIN 5: SECURITY POLICIES (RLS + CLS + MASKING)
-- ======================================================
USE CATALOG ecommerce_analytics_dev;
USE SCHEMA gold_layer;

# Define Filter & Masking Functions

In [0]:
%sql
-- 1. RLS: Row Level Security for Business Segmentation
-- This ensures Marketing only sees views/carts, and Finance only sees purchases.
CREATE OR REPLACE FUNCTION rls_event_filter(event_type STRING)
RETURNS BOOLEAN
RETURN
  is_account_group_member('admins') OR 
  is_account_group_member('ops_team') OR
  (is_account_group_member('finance_team') AND event_type = 'purchase') OR
  (is_account_group_member('marketing_team') AND event_type IN ('view','cart'));

In [0]:
%sql
-- 2. CLS: Masking Price for PII/Sensitive Data
-- Only Finance and Ops see the actual price; others see NULL.
CREATE OR REPLACE FUNCTION mask_price(p DOUBLE)
RETURNS DOUBLE
RETURN
  CASE
    WHEN is_account_group_member('finance_team') OR 
         is_account_group_member('ops_team') OR 
         is_account_group_member('admins') THEN p
    ELSE NULL
  END;


In [0]:
%sql
-- 3. CLS: User ID Masking (PII Protection)
-- Non-admin/ops users see a hashed string instead of the actual ID.
CREATE OR REPLACE FUNCTION mask_user(u STRING)
RETURNS STRING
RETURN
  CASE
    WHEN is_account_group_member('ops_team') OR 
         is_account_group_member('admins') THEN u
    ELSE sha1(u)
  END;

# Apply Policies to Your Actual Tables

In [0]:
%sql
-- Apply RLS to the Fact Sales table
ALTER TABLE ecommerce_analytics_dev.gold_layer.fact_sales 
SET ROW FILTER rls_event_filter ON (event_type);

-- Apply Column Masking to Price and User_ID
ALTER TABLE ecommerce_analytics_dev.gold_layer.fact_sales 
ALTER COLUMN price SET MASK gold_layer.mask_price;

ALTER TABLE ecommerce_analytics_dev.gold_layer.fact_sales 
ALTER COLUMN user_id SET MASK gold_layer.mask_user;


# Establish RBAC

In [0]:
%sql
-- 1. Grant Catalog Usage
GRANT USE CATALOG ON CATALOG ecommerce_analytics_dev TO `finance_team`;
GRANT USE CATALOG ON CATALOG ecommerce_analytics_dev TO `marketing_team`;
GRANT USE CATALOG ON CATALOG ecommerce_analytics_dev TO `ops_team`;

-- 2. Grant Schema Usage
GRANT USE SCHEMA ON SCHEMA ecommerce_analytics_dev.gold_layer TO `finance_team`;
GRANT USE SCHEMA ON SCHEMA ecommerce_analytics_dev.gold_layer TO `marketing_team`;
GRANT USE SCHEMA ON SCHEMA ecommerce_analytics_dev.gold_layer TO `ops_team`;

-- 3. Grant Table Selection
GRANT SELECT ON TABLE ecommerce_analytics_dev.gold_layer.fact_sales TO `finance_team`;
GRANT SELECT ON TABLE ecommerce_analytics_dev.gold_layer.fact_sales TO `marketing_team`;
GRANT SELECT ON TABLE ecommerce_analytics_dev.gold_layer.fact_sales TO `ops_team`;

GRANT SELECT ON TABLE ecommerce_analytics_dev.gold_layer.product_performance TO `finance_team`;
GRANT SELECT ON TABLE ecommerce_analytics_dev.gold_layer.product_performance TO `marketing_team`;
GRANT SELECT ON TABLE ecommerce_analytics_dev.gold_layer.product_performance TO `ops_team`;